# CTPG — Cross-Task Policy Guidance (NeurIPS 2024)
## HalfCheetah-MT5 · Explicit CTPG Implementation

**Implements explicitly:**
1. **Policy-Filter Gate** (§4.2): masks out control policies whose comparable guide Q-value < V-value of current task
2. **Guide-Block Gate** (§4.3): blocks guidance for tasks where log αᵢ > mean(log αⱼ) — i.e. already easy/mastered
3. **Hindsight Off-Policy Correction** (§4.1): re-labels guide action j_t → j'_t via maximum likelihood over stored actions

**Budget**: single config, 150k steps (~8-9 h on Colab T4). All outputs → Google Drive.

In [ ]:
import subprocess, sys
pkgs = [
    'mujoco==3.1.6',
    'gymnasium[mujoco]==0.29.1',
    'imageio',
    'imageio-ffmpeg',
    'tensorboard',
    'pandas',
    'tqdm',
]
for p in pkgs:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', p],
                       capture_output=True, text=True)
    print('  OK ', p) if r.returncode == 0 else print('  FAIL', p, r.stderr[-200:])

  OK  mujoco==3.1.6
  OK  gymnasium[mujoco]==0.29.1
  OK  imageio
  OK  imageio-ffmpeg
  OK  tensorboard
  OK  pandas
  OK  tqdm


IMPORTS & DRIVE MOUNT  

In [ ]:
import os, json, time, warnings, zipfile, tempfile, collections
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal
from torch.utils.tensorboard import SummaryWriter
from dataclasses import dataclass, asdict, field
from copy import deepcopy
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
warnings.filterwarnings('ignore')

os.environ['MUJOCO_GL']         = 'egl'
os.environ['PYOPENGL_PLATFORM'] = 'egl'

#Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

BASE     = '/content/drive/MyDrive/CTPG_v2'
CKPT_DIR = BASE + '/checkpoints'
TB_DIR   = BASE + '/tensorboard'
PLOT_DIR = BASE + '/plots'
VID_DIR  = BASE + '/videos'
LOG_DIR  = BASE + '/logs'
for d in [CKPT_DIR, TB_DIR, PLOT_DIR, VID_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

import mujoco, gymnasium
print(f'  Device   : {DEVICE}' + (f' ({torch.cuda.get_device_name(0)})' if torch.cuda.is_available() else ''))
print(f'  MuJoCo   : {mujoco.__version__}')
print(f'  Gymnasium: {gymnasium.__version__}')
print(f'  PyTorch  : {torch.__version__}')
print(f'  Outputs  : {BASE}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  Device   : cpu
  MuJoCo   : 3.1.6
  Gymnasium: 0.29.1
  PyTorch  : 2.10.0+cpu
  Outputs  : /content/drive/MyDrive/CTPG_v2


ENVIRONMENT  (HalfCheetah-MT5, gravity variants)

In [ ]:
import gymnasium as gym

GRAVITY_LIST = [4.91, 7.36, 9.81, 12.26, 14.72]
NUM_TASKS    = len(GRAVITY_LIST)
OBS_DIM      = 17
ACT_DIM      = 6

def make_hc_env(gravity, render_mode=None):
    env = gym.make('HalfCheetah-v4', render_mode=render_mode)
    env.unwrapped.model.opt.gravity[:] = [0, 0, -gravity]
    return env

class MultiTaskEnv:
    def __init__(self, seed=42):
        self.envs = []
        for i, g in enumerate(GRAVITY_LIST):
            env = make_hc_env(g)
            env.reset(seed=seed + i)
            self.envs.append(env)
        print(f'  {NUM_TASKS} tasks: gravity = {GRAVITY_LIST}')

    def reset(self, tid):
        obs, _ = self.envs[tid].reset()
        return obs.astype(np.float32)

    def step(self, tid, action):
        act = np.clip(action, -1.0, 1.0).astype(np.float32)
        obs, rew, term, trunc, info = self.envs[tid].step(act)
        return obs.astype(np.float32), float(rew), bool(term or trunc), info

    def close(self):
        for env in self.envs:
            try: env.close()
            except: pass

print('  Environment defined OK')

  Environment defined OK


NETWORKS (SAC + explicit CTPG components)

In [ ]:
LOG_STD_MIN, LOG_STD_MAX = -5, 2

def mlp(sizes, act=nn.ReLU, out_act=nn.Identity):
    layers = []
    for i in range(len(sizes) - 1):
        layers += [nn.Linear(sizes[i], sizes[i+1]),
                   (act if i < len(sizes)-2 else out_act)()]
    return nn.Sequential(*layers)


#Control Actor π_i
class Actor(nn.Module):
    """Shared-backbone multi-head SAC actor (MHSAC style).
    Each task i has its own output head; backbone is shared.
    """
    def __init__(self, obs_dim, act_dim, num_tasks, hidden=400):
        super().__init__()
        self.backbone = mlp([obs_dim, hidden, hidden])
        self.mu_heads  = nn.ModuleList([nn.Linear(hidden, act_dim) for _ in range(num_tasks)])
        self.ls_heads  = nn.ModuleList([nn.Linear(hidden, act_dim) for _ in range(num_tasks)])

    def forward(self, obs, tid):
        """tid: (B,) long tensor of task indices."""
        h   = self.backbone(obs)
        mu  = torch.stack([self.mu_heads[t](h[b:b+1]) for b, t in enumerate(tid.tolist())], 0).squeeze(1)
        ls  = torch.stack([self.ls_heads[t](h[b:b+1]) for b, t in enumerate(tid.tolist())], 0).squeeze(1)
        return mu, ls.clamp(LOG_STD_MIN, LOG_STD_MAX)

    def forward_all_tasks(self, obs):
        """Return mu and std for ALL tasks. obs: (B, obs_dim).
        Returns AM: (B, N, act_dim), AS: (B, N, act_dim).
        Used for hindsight off-policy correction.
        """
        h = self.backbone(obs)
        mus, stds = [], []
        for t in range(len(self.mu_heads)):
            mu = self.mu_heads[t](h)
            ls = self.ls_heads[t](h).clamp(LOG_STD_MIN, LOG_STD_MAX)
            mus.append(mu); stds.append(ls.exp())
        return torch.stack(mus, 1), torch.stack(stds, 1)

    def sample(self, obs, tid):
        mu, ls = self.forward(obs, tid)
        dist   = Normal(mu, ls.exp())
        xt     = dist.rsample()
        yt     = torch.tanh(xt)
        lp     = dist.log_prob(xt) - torch.log(1 - yt.pow(2) + 1e-6)
        return yt, lp.sum(-1, keepdim=True), torch.tanh(mu)

    def log_prob_action(self, obs, act, tid):
        mu, ls = self.forward(obs, tid)
        dist   = Normal(mu, ls.exp())
        act_c  = act.clamp(-1+1e-6, 1-1e-6)
        xt     = torch.atanh(act_c)
        lp     = dist.log_prob(xt) - torch.log(1 - act_c.pow(2) + 1e-6)
        return lp.sum(-1)


#Control Critic Q_i
class Critic(nn.Module):
    def __init__(self, obs_dim, act_dim, num_tasks, hidden=400):
        super().__init__()
        d = obs_dim + act_dim
        self.backbone1 = mlp([d, hidden, hidden])
        self.backbone2 = mlp([d, hidden, hidden])
        self.q1_heads  = nn.ModuleList([nn.Linear(hidden, 1) for _ in range(num_tasks)])
        self.q2_heads  = nn.ModuleList([nn.Linear(hidden, 1) for _ in range(num_tasks)])

    def forward(self, obs, act, tid):
        x   = torch.cat([obs, act], -1)
        h1  = self.backbone1(x)
        h2  = self.backbone2(x)
        q1  = torch.stack([self.q1_heads[t](h1[b:b+1]) for b, t in enumerate(tid.tolist())], 0).squeeze(1)
        q2  = torch.stack([self.q2_heads[t](h2[b:b+1]) for b, t in enumerate(tid.tolist())], 0).squeeze(1)
        return q1, q2


#Guide Actor Π^g_i
class GuideActor(nn.Module):
    def __init__(self, obs_dim, num_tasks, hidden=400):
        super().__init__()
        self.backbone = mlp([obs_dim, hidden, hidden])
        self.heads = nn.ModuleList([nn.Linear(hidden, num_tasks) for _ in range(num_tasks)])

    def logits(self, obs, tid):
        h = self.backbone(obs)
        return torch.stack([self.heads[t](h[b:b+1]) for b, t in enumerate(tid.tolist())], 0).squeeze(1)

    def probs(self, obs, tid, mask=None):
        lgts = self.logits(obs, tid)
        if mask is not None:
            lgts = lgts + (mask.float().log() + 1e-45)
        return F.softmax(lgts, dim=-1)

    def sample(self, obs, tid, mask=None):
        probs = self.probs(obs, tid, mask)
        dist  = torch.distributions.Categorical(probs)
        j     = dist.sample()
        lp    = dist.log_prob(j)
        return j, lp, probs


#Comparable Guide Critic Q̂^g_i
class ComparableGuideCritic(nn.Module):
    """Estimates expected return using CURRENT task's entropy (Eq. 13).
    Allows direct comparison with V_i(s) from SAC (Appendix B).
    Multi-head: one head per task.
    """
    def __init__(self, obs_dim, num_tasks, hidden=400):
        super().__init__()
        self.backbone = mlp([obs_dim, hidden, hidden])
        self.heads = nn.ModuleList([nn.Linear(hidden, num_tasks) for _ in range(num_tasks)])

    def forward(self, obs, tid):
        h   = self.backbone(obs)
        out = torch.stack([self.heads[t](h[b:b+1]) for b, t in enumerate(tid.tolist())], 0).squeeze(1)
        return out


#Guide Critic Q^g_i  (standard SAC critic for guide)
class GuideCritic(nn.Module):
    """Standard Q^g_i(s, j) using maximum entropy objective for guide SAC."""
    def __init__(self, obs_dim, num_tasks, hidden=400):
        super().__init__()
        self.backbone = mlp([obs_dim, hidden, hidden])
        self.heads    = nn.ModuleList([nn.Linear(hidden, num_tasks) for _ in range(num_tasks)])

    def forward(self, obs, tid):
        """Returns Q^g_i(s, :) shape (B, N)."""
        h   = self.backbone(obs)
        out = torch.stack([self.heads[t](h[b:b+1]) for b, t in enumerate(tid.tolist())], 0).squeeze(1)
        return out  # (B, N)


print('  Networks defined OK')

  Networks defined OK


REPLAY BUFFERS

In [ ]:
class ReplayBuffer:
    """Standard SAC replay buffer for control policy transitions."""
    def __init__(self, obs_dim, act_dim, capacity=500_000, device=DEVICE):
        self.cap  = capacity; self.ptr = 0; self.size = 0; self.dev = device
        self.o    = np.zeros((capacity, obs_dim),  np.float32)
        self.a    = np.zeros((capacity, act_dim),  np.float32)
        self.r    = np.zeros((capacity, 1),        np.float32)
        self.no   = np.zeros((capacity, obs_dim),  np.float32)
        self.done = np.zeros((capacity, 1),        np.float32)
        self.tid  = np.zeros(capacity,             np.int64)

    def add(self, o, a, r, no, done, tid):
        i = self.ptr
        self.o[i]=o; self.a[i]=a; self.r[i]=r
        self.no[i]=no; self.done[i]=done; self.tid[i]=tid
        self.ptr  = (i + 1) % self.cap
        self.size = min(self.size + 1, self.cap)

    def sample(self, bs):
        idx = np.random.randint(0, self.size, bs)
        f   = lambda x: torch.tensor(x[idx], device=self.dev)
        return f(self.o), f(self.a), f(self.r), f(self.no), f(self.done), f(self.tid)

    def __len__(self): return self.size


class GuideReplayBuffer:
    """Guide policy buffer. Stores K-step segments for hindsight correction.
    Each entry: (task_i, s_t, j_t, K actions {a_t'}, K states {s_t'}, K rewards, s_{t+K}).
    """
    def __init__(self, obs_dim, act_dim, K, num_tasks, capacity=100_000, device=DEVICE):
        self.cap = capacity; self.ptr = 0; self.size = 0
        self.dev = device; self.K = K; self.N = num_tasks
        self.s0   = np.zeros((capacity, obs_dim),     np.float32)  # s_t
        self.j    = np.zeros(capacity,                np.int64)    # j_t (guide action)
        self.segs = np.zeros((capacity, K, act_dim),  np.float32)  # {a_t'}
        self.sobs = np.zeros((capacity, K, obs_dim),  np.float32)  # {s_t'}
        self.rews = np.zeros((capacity, K),           np.float32)  # {r_t'}
        self.sK   = np.zeros((capacity, obs_dim),     np.float32)  # s_{t+K}
        self.tid  = np.zeros(capacity,                np.int64)    # task i
        self.done = np.zeros(capacity,                np.float32)  # episode done within K steps

    def add(self, tid, s0, j, seg_acts, seg_obs, seg_rews, sK, done):
        i = self.ptr
        self.s0[i]   = s0
        self.j[i]    = j
        self.segs[i] = seg_acts[:self.K]
        self.sobs[i] = seg_obs[:self.K]
        self.rews[i] = seg_rews[:self.K]
        self.sK[i]   = sK
        self.tid[i]  = tid
        self.done[i] = done
        self.ptr  = (i + 1) % self.cap
        self.size = min(self.size + 1, self.cap)

    def sample(self, bs):
        idx    = np.random.randint(0, self.size, bs)
        f      = lambda x: torch.tensor(x[idx], device=self.dev)
        return (f(self.tid).long(), f(self.s0),
                f(self.j).long(),   f(self.segs),
                f(self.sobs),       f(self.rews),
                f(self.sK),         f(self.done))

    def __len__(self): return self.size


print('  Replay buffers defined OK')

  Replay buffers defined OK


CTPG AGENT (all three gates + hindsight)

In [ ]:
@dataclass
class Config:
    # SAC hyper-params
    hidden      : int   = 400
    actor_lr    : float = 1e-4
    critic_lr   : float = 1e-4
    alpha_lr    : float = 1e-4
    gamma       : float = 0.99
    tau         : float = 0.005
    batch_size  : int   = 256
    # CTPG hyper-params
    guide_step  : int   = 10      # guide chooses behavior policy every K steps
    guide_lr    : float = 1e-4
    guide_alpha : float = 0.05    # entropy coefficient for guide SAC
    mc_samples  : int   = 5       # H — Monte Carlo samples for V-value estimation
    # Training schedule
    total_steps : int   = 150_000
    warmup      : int   = 5_000
    eval_freq   : int   = 10_000
    eval_eps    : int   = 5
    ckpt_freq   : int   = 25_000
    log_freq    : int   = 1_000
    seed        : int   = 42


class CTPGAgent:
    def __init__(self, cfg: Config, dev=DEVICE):
        self.cfg = cfg; self.dev = dev; self.N = NUM_TASKS

        # Control networks
        self.actor  = Actor(OBS_DIM, ACT_DIM, NUM_TASKS, cfg.hidden).to(dev)
        self.critic = Critic(OBS_DIM, ACT_DIM, NUM_TASKS, cfg.hidden).to(dev)
        self.ctgt   = deepcopy(self.critic);
        for p in self.ctgt.parameters(): p.requires_grad_(False)

        # Per-task SAC temperature α_i
        self.log_alpha = nn.ParameterList([
            nn.Parameter(torch.zeros(1, device=dev)) for _ in range(NUM_TASKS)
        ])
        self.tgt_ent = -float(ACT_DIM)

        #Guide networks
        self.guide_actor   = GuideActor(OBS_DIM, NUM_TASKS, cfg.hidden).to(dev)
        self.guide_critic  = GuideCritic(OBS_DIM, NUM_TASKS, cfg.hidden).to(dev)
        self.guide_ctgt    = deepcopy(self.guide_critic)
        for p in self.guide_ctgt.parameters(): p.requires_grad_(False)
        self.cmp_critic    = ComparableGuideCritic(OBS_DIM, NUM_TASKS, cfg.hidden).to(dev)

        # Guide per-task temperature
        self.guide_log_alpha = nn.ParameterList([
            nn.Parameter(torch.zeros(1, device=dev)) for _ in range(NUM_TASKS)
        ])

        # Optimizers
        self.oa  = torch.optim.Adam(self.actor.parameters(),       lr=cfg.actor_lr)
        self.oc  = torch.optim.Adam(self.critic.parameters(),      lr=cfg.critic_lr)
        self.ola = [torch.optim.Adam([la], lr=cfg.alpha_lr) for la in self.log_alpha]
        self.og  = torch.optim.Adam(
            list(self.guide_actor.parameters()) +
            list(self.guide_critic.parameters()) +
            list(self.cmp_critic.parameters()),
            lr=cfg.guide_lr
        )
        self.ogla = [torch.optim.Adam([la], lr=cfg.guide_lr) for la in self.guide_log_alpha]

    # Helpers
    def alpha(self, tid):
        return self.log_alpha[tid].exp().item()

    def guide_alpha_val(self, tid):
        return self.guide_log_alpha[tid].exp().item()

    @torch.no_grad()
    def act(self, obs, tid, det=False):
        o = torch.FloatTensor(obs).unsqueeze(0).to(self.dev)
        t = torch.tensor([tid], dtype=torch.long, device=self.dev)
        if det:
            mu, _ = self.actor.forward(o, t)
            return torch.tanh(mu).squeeze(0).cpu().numpy()
        a, _, _ = self.actor.sample(o, t)
        return a.squeeze(0).cpu().numpy()

    # Guide-Block Gate
    def guide_block_gate(self):
        log_alphas = torch.stack([la.detach() for la in self.log_alpha]).squeeze(-1)
        mean_la    = log_alphas.mean()
        needs_guide = (log_alphas <= mean_la).tolist()  # bool list length N
        return needs_guide   # needs_guide[i] = True → task i needs guidance

    # Policy-Filter Gate
    @torch.no_grad()
    def policy_filter_gate(self, obs_t, tid):
        o = torch.FloatTensor(obs_t).unsqueeze(0).to(self.dev)
        t = torch.tensor([tid], dtype=torch.long, device=self.dev)

        # Monte Carlo with H samples
        v_samples = []
        for _ in range(self.cfg.mc_samples):
            a_mc, lp_mc, _ = self.actor.sample(o, t)
            q1, q2 = self.critic(o, a_mc, t)
            q_mc   = torch.min(q1, q2)
            # In SAC: V_i(s) = E[Q_i - α_i log π_i]
            v_samples.append(q_mc - self.log_alpha[tid].exp() * lp_mc)
        v_i = torch.stack(v_samples, 0).mean(0)   # (1, 1)

        # Q̂^g_i(s, :) shape (1, N)
        qhat = self.cmp_critic(o, t)   # (1, N)

        mask = (qhat >= v_i).squeeze(0)
        # If nothing passes, fall back to self-policy
        if not mask.any():
            mask[tid] = True
        return mask

    # Choose behavior policy
    @torch.no_grad()
    def choose_behavior_policy(self, obs_t, tid, needs_guide):
        if not needs_guide:
            return tid, False

        # Policy-filter gate
        mask = self.policy_filter_gate(obs_t, tid)

        o = torch.FloatTensor(obs_t).unsqueeze(0).to(self.dev)
        t = torch.tensor([tid], dtype=torch.long, device=self.dev)

        # Sample from guide with mask applied
        j, _, _ = self.guide_actor.sample(o, t, mask=mask.unsqueeze(0))
        return j.item(), True

    # Hindsight Off-Policy Correction
    def hindsight_correct(self, seg_obs, seg_acts):
        B, K, _ = seg_acts.shape
        log_probs = torch.zeros(B, self.N, device=self.dev)   # (B, N)

        for k in range(K):
            obs_k = seg_obs[:, k, :]    # (B, obs_dim)
            act_k = seg_acts[:, k, :]   # (B, act_dim)
            for j in range(self.N):
                tj = torch.full((B,), j, dtype=torch.long, device=self.dev)
                lp = self.actor.log_prob_action(obs_k, act_k, tj)  # (B,)
                log_probs[:, j] = log_probs[:, j] + lp

        j_prime = log_probs.argmax(dim=-1)   # (B,)
        return j_prime

    # Update control policy
    def update_control(self, batch):
        obs, acts, rews, nobs, dones, tids = batch
        cfg = self.cfg

        # Critic loss (Eq.1)
        with torch.no_grad():
            na, nlp, _ = self.actor.sample(nobs, tids)
            alpha_vec  = torch.tensor([self.log_alpha[t].exp().item() for t in tids.tolist()],
                                       device=self.dev).unsqueeze(-1)
            q1t, q2t   = self.ctgt(nobs, na, tids)
            qt         = torch.min(q1t, q2t) - alpha_vec * nlp
            yt         = rews + cfg.gamma * (1 - dones) * qt

        q1, q2 = self.critic(obs, acts, tids)
        lc     = F.mse_loss(q1, yt) + F.mse_loss(q2, yt)
        self.oc.zero_grad(); lc.backward(); self.oc.step()

        # Actor loss (Eq.3)
        pi, lp, _ = self.actor.sample(obs, tids)
        q1p, q2p  = self.critic(obs, pi, tids)
        alpha_vec2 = torch.tensor([self.log_alpha[t].exp().item() for t in tids.tolist()],
                                   device=self.dev).unsqueeze(-1)
        la = (alpha_vec2 * lp - torch.min(q1p, q2p)).mean()
        self.oa.zero_grad(); la.backward()
        nn.utils.clip_grad_norm_(self.actor.parameters(), 1.0)
        self.oa.step()

        # Per-task alpha update (Eq.4)
        with torch.no_grad():
            _, lp2, _ = self.actor.sample(obs, tids)
        for i in range(self.N):
            mask_i = (tids.squeeze(-1) == i) if tids.dim() > 1 else (tids == i)
            if mask_i.any():
                lp_i   = lp2[mask_i]
                ll     = -(self.log_alpha[i] * (lp_i + self.tgt_ent).detach()).mean()
                self.ola[i].zero_grad(); ll.backward(); self.ola[i].step()

        # Soft target update
        for p, tp in zip(self.critic.parameters(), self.ctgt.parameters()):
            tp.data.lerp_(p.data, cfg.tau)

        return {'critic': lc.item(), 'actor': la.item(),
                'alpha': np.mean([self.alpha(i) for i in range(self.N)])}

    # Update guide policy
    def update_guide(self, guide_batch, needs_guide):
        tids, s0, j_t, seg_acts, seg_obs, seg_rews, sK, done_flag = guide_batch
        cfg = self.cfg

        # Only train on tasks in T^g (guide-block gate at batch level)
        tg_mask = torch.tensor([needs_guide[t.item()] for t in tids], device=self.dev)
        if not tg_mask.any():
            return {'guide_loss': 0.0}

        s0_g   = s0[tg_mask];     tids_g  = tids[tg_mask]
        sa_g   = seg_acts[tg_mask]; so_g  = seg_obs[tg_mask]
        sr_g   = seg_rews[tg_mask]; sK_g  = sK[tg_mask]
        done_g = done_flag[tg_mask]

        # Hindsight Off-Policy Correction
        with torch.no_grad():
            j_prime = self.hindsight_correct(so_g, sa_g)   # (B_g,)

        # Guide reward
        K     = cfg.guide_step
        gamma = cfg.gamma
        disc  = torch.tensor([gamma**k for k in range(K)], device=self.dev)
        r_guide = (sr_g * disc.unsqueeze(0)).sum(-1, keepdim=True)  # (B_g, 1)

        # critic update
        with torch.no_grad():
            # V^g_i(s_{t+K}): sample from guide actor and take min Q^g
            j_next, lp_next, _ = self.guide_actor.sample(sK_g, tids_g)
            qg_next    = self.guide_ctgt(sK_g, tids_g)  # (B_g, N)
            ga_vec     = torch.tensor(
                [self.guide_alpha_val(t.item()) for t in tids_g], device=self.dev).unsqueeze(-1)
            vg_next    = qg_next.gather(1, j_next.unsqueeze(-1)) - ga_vec * lp_next.unsqueeze(-1)
            yt_g       = r_guide + (gamma**K) * (1 - done_g.unsqueeze(-1)) * vg_next

        qg_pred = self.guide_critic(s0_g, tids_g)                      # (B_g, N)
        qg_pred_j = qg_pred.gather(1, j_prime.unsqueeze(-1))            # (B_g, 1)
        lc_g = F.mse_loss(qg_pred_j, yt_g)

        # (comparable) critic update
        with torch.no_grad():
            # Comparable target: uses current task's entropy α_i H(π_i)
            alpha_i_vec = torch.tensor(
                [self.alpha(t.item()) for t in tids_g], device=self.dev).unsqueeze(-1)
            # Entropy of control policy at next state
            _, lp_ctrl_next, _ = self.actor.sample(sK_g, tids_g)
            h_ctrl_next = -lp_ctrl_next  # entropy approx
            cmp_next = self.cmp_critic(sK_g, tids_g).gather(1, j_next.unsqueeze(-1))
            yt_cmp   = r_guide + alpha_i_vec * h_ctrl_next + \
                       (gamma**K) * (1 - done_g.unsqueeze(-1)) * cmp_next

        cmp_pred   = self.cmp_critic(s0_g, tids_g).gather(1, j_prime.unsqueeze(-1))
        lc_cmp     = F.mse_loss(cmp_pred, yt_cmp)

        # Guide actor update (discrete SAC)
        j_samp, lp_guide, probs_g = self.guide_actor.sample(s0_g, tids_g)
        qg_vals = self.guide_critic(s0_g, tids_g).detach()   # (B_g, N)
        ga_vec2 = torch.tensor(
            [self.guide_alpha_val(t.item()) for t in tids_g], device=self.dev).unsqueeze(-1)
        # E_{j~Π^g}[α^g log Π^g(j|s) - Q^g(s,j)]
        la_g = (probs_g * (ga_vec2 * torch.log(probs_g + 1e-8) - qg_vals)).sum(-1).mean()

        # Total guide loss
        guide_loss = lc_g + lc_cmp + la_g
        self.og.zero_grad(); guide_loss.backward()
        nn.utils.clip_grad_norm_(
            list(self.guide_actor.parameters()) +
            list(self.guide_critic.parameters()) +
            list(self.cmp_critic.parameters()), 1.0)
        self.og.step()

        # Guide alpha update (per-task)
        with torch.no_grad():
            _, lp_gg, _ = self.guide_actor.sample(s0_g, tids_g)
        tgt_ent_g = -np.log(1.0 / self.N) * 0.5   # half of uniform entropy
        for i in range(self.N):
            mask_i = (tids_g == i)
            if mask_i.any() and needs_guide[i]:
                lp_i = lp_gg[mask_i]
                ll_g = -(self.guide_log_alpha[i] * (lp_i + tgt_ent_g).detach()).mean()
                self.ogla[i].zero_grad(); ll_g.backward(); self.ogla[i].step()

        # Guide critic soft target update
        for p, tp in zip(self.guide_critic.parameters(), self.guide_ctgt.parameters()):
            tp.data.lerp_(p.data, self.cfg.tau)

        return {'guide_critic': lc_g.item(), 'cmp_critic': lc_cmp.item(),
                'guide_actor': la_g.item(), 'guide_loss': guide_loss.item()}

    # Checkpoint save/load
    def save(self, path, step):
        ck = {
            'step': step, 'cfg': asdict(self.cfg),
            'actor':         self.actor.state_dict(),
            'critic':        self.critic.state_dict(),
            'ctgt':          self.ctgt.state_dict(),
            'guide_actor':   self.guide_actor.state_dict(),
            'guide_critic':  self.guide_critic.state_dict(),
            'guide_ctgt':    self.guide_ctgt.state_dict(),
            'cmp_critic':    self.cmp_critic.state_dict(),
            'log_alpha':     [la.item() for la in self.log_alpha],
            'guide_log_alpha': [la.item() for la in self.guide_log_alpha],
        }
        torch.save(ck, path)
        print(f'    [SAVED] {os.path.basename(path)}')

    @classmethod
    def load(cls, path, dev=DEVICE):
        ck  = torch.load(path, map_location=dev)
        ag  = cls(Config(**ck['cfg']), dev)
        ag.actor.load_state_dict(ck['actor'])
        ag.critic.load_state_dict(ck['critic'])
        ag.ctgt.load_state_dict(ck['ctgt'])
        ag.guide_actor.load_state_dict(ck['guide_actor'])
        ag.guide_critic.load_state_dict(ck['guide_critic'])
        ag.guide_ctgt.load_state_dict(ck['guide_ctgt'])
        ag.cmp_critic.load_state_dict(ck['cmp_critic'])
        for i, v in enumerate(ck['log_alpha']):
            ag.log_alpha[i].data.fill_(v)
        for i, v in enumerate(ck['guide_log_alpha']):
            ag.guide_log_alpha[i].data.fill_(v)
        return ag, ck['step']


print('  CTPGAgent defined OK')

  CTPGAgent defined OK


TRAINING LOOP (Algorithm 3 — full CTPG)

In [ ]:
from tqdm import tqdm

def evaluate(agent, envs, n_eps=5):
    """Run n_eps episodes per task, return mean + per-task rewards."""
    per_task = []
    for tid in range(NUM_TASKS):
        ep_rewards = []
        for _ in range(n_eps):
            obs = envs.reset(tid); total_r = 0.0
            for _ in range(1000):
                act = agent.act(obs, tid, det=True)
                obs, r, done, _ = envs.step(tid, act)
                total_r += r
                if done: break
            ep_rewards.append(total_r)
        per_task.append(np.mean(ep_rewards))
    return np.mean(per_task), per_task


def train():
    cfg   = Config()
    torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)

    # Check for existing checkpoint
    run_id    = 'ctpg_mt5_explicit'
    ckpt_path = os.path.join(CKPT_DIR, run_id)
    os.makedirs(ckpt_path, exist_ok=True)
    resume_path = os.path.join(ckpt_path, 'latest.pt')

    agent   = CTPGAgent(cfg)
    start_step = 0
    if os.path.exists(resume_path):
        try:
            agent, start_step = CTPGAgent.load(resume_path)
            print(f'  Resumed from step {start_step:,}')
        except Exception as e:
            print(f'  Resume failed ({e}), starting fresh')
            agent = CTPGAgent(cfg); start_step = 0

    envs    = MultiTaskEnv(seed=cfg.seed)
    rbuf    = ReplayBuffer(OBS_DIM, ACT_DIM, capacity=500_000)
    gbuf    = GuideReplayBuffer(OBS_DIM, ACT_DIM, K=cfg.guide_step,
                                 num_tasks=NUM_TASKS, capacity=100_000)

    writer  = SummaryWriter(log_dir=os.path.join(TB_DIR, run_id))

    # Per-task episode state
    obs_list   = [envs.reset(i) for i in range(NUM_TASKS)]
    ep_rewards = [0.0] * NUM_TASKS
    ep_steps   = [0]   * NUM_TASKS
    ep_count   = [0]   * NUM_TASKS

    # Guide step tracking: per-task segment buffers
    seg_buf = [{'acts': [], 'obs': [], 'rews': [], 's0': None, 'j': None}
               for _ in range(NUM_TASKS)]

    best_eval  = -1e9
    eval_hist  = []
    loss_hist  = []
    start_time = time.time()

    print('='*70)
    print('  CTPG-MT5 Training   (explicit policy-filter + guide-block + hindsight)')
    print(f'  Steps: {cfg.total_steps:,}  |  K={cfg.guide_step}  |  MC={cfg.mc_samples}')
    print('='*70)
    print(f'  {"Step":>8}  {"Task":>4}  {"EpRew":>9}  {"EvalMean":>10}  {"BestEval":>10}  {"Time":>7}')
    print('  ' + '-'*60)

    for step in range(start_step, cfg.total_steps):

        # Round-robin task selection
        tid = step % NUM_TASKS
        obs = obs_list[tid]

        # Guide-Block Gate: compute
        needs_guide = agent.guide_block_gate()

        # Every K steps: choose behavior policy
        if ep_steps[tid] % cfg.guide_step == 0:
            bhv_tid, from_guide = agent.choose_behavior_policy(
                obs, tid, needs_guide[tid])
            # Start new K-step segment
            seg_buf[tid] = {'acts': [], 'obs': [], 'rews': [],
                             's0': obs.copy(), 'j': bhv_tid,
                             'from_guide': from_guide}
        else:
            bhv_tid = seg_buf[tid].get('j', tid)

        # Take action with behavior policy
        if step < cfg.warmup:
            act = envs.envs[tid].action_space.sample().astype(np.float32)
        else:
            act = agent.act(obs, bhv_tid)  # use behavior policy π_{j_t}

        next_obs, rew, done, _ = envs.step(tid, act)

        # Store control transition with TASK i's reward (not behavior task's)
        # The stored task id is i (current task), but action was from behavior policy j
        rbuf.add(obs, act, rew, next_obs, float(done), tid)

        # Accumulate K-step segment for guide buffer
        sb = seg_buf[tid]
        sb['acts'].append(act.copy())
        sb['obs'].append(obs.copy())
        sb['rews'].append(rew)

        # When segment is complete (K steps) or episode ends → push to guide buffer
        seg_done = (len(sb['acts']) >= cfg.guide_step) or done
        if seg_done and sb.get('from_guide', False) and sb['s0'] is not None:
            K_actual = len(sb['acts'])
            # Pad if episode ended early
            pad = cfg.guide_step - K_actual
            acts_arr = np.array(sb['acts'] + [sb['acts'][-1]] * pad, np.float32)
            obs_arr  = np.array(sb['obs']  + [next_obs]        * pad, np.float32)
            rews_arr = np.array(sb['rews'] + [0.0]             * pad, np.float32)
            gbuf.add(
                tid, sb['s0'], sb['j'],
                acts_arr, obs_arr, rews_arr,
                next_obs, float(done)
            )

        ep_rewards[tid] += rew
        ep_steps[tid]   += 1

        if done:
            obs_list[tid] = envs.reset(tid)
            ep_count[tid] += 1
            if step % cfg.log_freq < NUM_TASKS:
                elapsed = (time.time() - start_time) / 60
                print(f'  {step:>8,}  {tid:>4}  {ep_rewards[tid]:>9.1f}  '
                      f'{"---":>10}  {best_eval:>10.1f}  {elapsed:>6.1f}m')
            ep_rewards[tid] = 0.0
            ep_steps[tid]   = 0
        else:
            obs_list[tid] = next_obs

        # Training
        if step >= cfg.warmup and len(rbuf) >= cfg.batch_size:
            batch   = rbuf.sample(cfg.batch_size)
            c_losses = agent.update_control(batch)

            # Guide update every K steps
            g_losses = {}
            if step % cfg.guide_step == 0 and len(gbuf) >= cfg.batch_size // 4:
                gbatch   = gbuf.sample(max(32, cfg.batch_size // 4))
                g_losses = agent.update_guide(gbatch, needs_guide)

            # TensorBoard
            if step % cfg.log_freq == 0:
                writer.add_scalar('control/critic_loss', c_losses['critic'], step)
                writer.add_scalar('control/actor_loss',  c_losses['actor'],  step)
                writer.add_scalar('control/mean_alpha',  c_losses['alpha'],  step)
                for i in range(NUM_TASKS):
                    writer.add_scalar(f'alpha/task_{i}', agent.alpha(i), step)
                    writer.add_scalar(f'guide_block/needs_guide_task_{i}',
                                       float(needs_guide[i]), step)
                if g_losses:
                    writer.add_scalar('guide/loss',         g_losses.get('guide_loss', 0), step)
                    writer.add_scalar('guide/critic_loss',  g_losses.get('guide_critic', 0), step)
                    writer.add_scalar('guide/cmp_critic',   g_losses.get('cmp_critic', 0), step)
                    writer.add_scalar('guide/actor_loss',   g_losses.get('guide_actor', 0), step)
                loss_hist.append({'step': step, **c_losses, **g_losses})

        # Evaluation
        if step % cfg.eval_freq == 0 and step >= cfg.warmup:
            mean_r, per_task = evaluate(agent, envs, cfg.eval_eps)
            elapsed = (time.time() - start_time) / 60
            print(f'  {step:>8,}   EVAL   ALL  {"---":>9}  {mean_r:>10.1f}  '
                  f'{best_eval:>10.1f}  {elapsed:>6.1f}m')
            print(f'           per-task: {[f"{r:.0f}" for r in per_task]}')

            # Log guide-block gate status
            ng = agent.guide_block_gate()
            print(f'           guide-block gate T^g: {[i for i,v in enumerate(ng) if v]}')

            writer.add_scalar('eval/mean_reward', mean_r, step)
            for i, r in enumerate(per_task):
                writer.add_scalar(f'eval/task_{i}_reward', r, step)

            eval_hist.append({'step': step, 'mean': mean_r, 'per_task': per_task})

            if mean_r > best_eval:
                best_eval = mean_r
                agent.save(os.path.join(ckpt_path, 'best.pt'), step)
                writer.add_scalar('eval/best_reward', best_eval, step)

        # Periodic checkpoint
        if step % cfg.ckpt_freq == 0 and step > start_step:
            agent.save(resume_path, step)
            log_data = {'cfg': asdict(cfg), 'eval_hist': eval_hist,
                        'loss_hist': loss_hist[-500:], 'best_eval': best_eval}
            with open(os.path.join(LOG_DIR, run_id + '.json'), 'w') as f:
                json.dump(log_data, f, indent=2)

    #Final save
    agent.save(resume_path, cfg.total_steps)
    log_data = {'cfg': asdict(cfg), 'eval_hist': eval_hist,
                'loss_hist': loss_hist[-1000:], 'best_eval': best_eval}
    with open(os.path.join(LOG_DIR, run_id + '.json'), 'w') as f:
        json.dump(log_data, f, indent=2)

    writer.close()
    envs.close()

    print('='*70)
    print(f'  TRAINING COMPLETE   Best eval: {best_eval:.1f}')
    print('='*70)
    return agent, eval_hist, loss_hist, run_id



#Run
agent, eval_hist, loss_hist, run_id = train()

  Resumed from step 150,000
  5 tasks: gravity = [4.91, 7.36, 9.81, 12.26, 14.72]
  CTPG-MT5 Training   (explicit policy-filter + guide-block + hindsight)
  Steps: 150,000  |  K=10  |  MC=5
      Step  Task      EpRew    EvalMean    BestEval     Time
  ------------------------------------------------------------
    [SAVED] latest.pt
  TRAINING COMPLETE   Best eval: -1000000000.0


 PLOTS

In [ ]:
PAL = ['#58a6ff','#f78166','#56d364','#e3b341','#bc8cff']

if eval_hist:
    # Fig 1 — Mean eval reward over training
    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor('#0d1117'); ax.set_facecolor('#161b22')
    xs = [e['step'] for e in eval_hist]
    ys = [e['mean'] for e in eval_hist]
    ax.plot(xs, ys, color='#58a6ff', lw=2, label='Mean Eval Reward')
    ax.axhline(max(ys), color='#f0c040', ls='--', lw=1, label=f'Best={max(ys):.1f}')
    ax.set_xlabel('Steps', color='#c9d1d9'); ax.set_ylabel('Eval Reward', color='#c9d1d9')
    ax.set_title('CTPG-MT5 — Mean Eval Reward', color='white', fontweight='bold')
    ax.legend(); ax.grid(True, alpha=0.3)
    ax.tick_params(colors='#c9d1d9')
    p1 = PLOT_DIR + '/Fig1_mean_reward.png'
    plt.savefig(p1, dpi=150, bbox_inches='tight'); plt.close()
    print(f'  Saved: {p1}')

    # Fig 2 — Per-task learning curves
    fig, ax = plt.subplots(figsize=(12, 5))
    fig.patch.set_facecolor('#0d1117'); ax.set_facecolor('#161b22')
    for t in range(NUM_TASKS):
        ys_t = [e['per_task'][t] for e in eval_hist]
        ax.plot(xs, ys_t, color=PAL[t], lw=1.5, alpha=0.8, label=f'g={GRAVITY_LIST[t]}')
    ax.plot(xs, ys, 'w-', lw=2.5, zorder=5, label='Mean')
    ax.set_xlabel('Steps', color='#c9d1d9'); ax.set_ylabel('Eval Reward', color='#c9d1d9')
    ax.set_title('CTPG-MT5 — Per-Task Learning Curves', color='white', fontweight='bold')
    ax.legend(fontsize=8, ncol=2); ax.grid(True, alpha=0.3)
    ax.tick_params(colors='#c9d1d9')
    p2 = PLOT_DIR + '/Fig2_per_task.png'
    plt.savefig(p2, dpi=150, bbox_inches='tight'); plt.close()
    print(f'  Saved: {p2}')

if loss_hist:
    # Fig 3 — Loss curves
    keys  = ['critic', 'actor', 'guide_loss', 'alpha']
    cols  = ['#58a6ff', '#f78166', '#56d364', '#e3b341']
    fig, axes = plt.subplots(1, 4, figsize=(20, 4))
    fig.patch.set_facecolor('#0d1117')
    fig.suptitle('CTPG-MT5 — Loss Curves', color='white', fontweight='bold', fontsize=13)
    for ax, key, col in zip(axes, keys, cols):
        ax.set_facecolor('#161b22')
        xs_l = [e['step'] for e in loss_hist if key in e]
        ys_l = [e[key]    for e in loss_hist if key in e]
        if xs_l:
            ax.plot(xs_l, ys_l, color=col, lw=1, alpha=0.4)
            if len(ys_l) > 20:
                w  = max(1, len(ys_l) // 20)
                sm = np.convolve(ys_l, np.ones(w)/w, mode='valid')
                ax.plot(xs_l[w-1:], sm, color='white', lw=2)
        ax.set_title(key, color='white', fontweight='bold')
        ax.set_xlabel('Steps', color='#c9d1d9'); ax.grid(True, alpha=0.3)
        ax.tick_params(colors='#c9d1d9')
    plt.tight_layout()
    p3 = PLOT_DIR + '/Fig3_losses.png'
    plt.savefig(p3, dpi=150, bbox_inches='tight'); plt.close()
    print(f'  Saved: {p3}')

# Fig 4 — Final bar chart per task
if eval_hist:
    final = eval_hist[-1]
    fig, ax = plt.subplots(figsize=(9, 5))
    fig.patch.set_facecolor('#0d1117'); ax.set_facecolor('#161b22')
    labels = [f'g={g}' for g in GRAVITY_LIST]
    bars   = ax.bar(labels, final['per_task'], color=PAL, alpha=0.9, edgecolor='#30363d')
    ax.axhline(final['mean'], color='#f0c040', ls='--', lw=1.5, label=f"Mean={final['mean']:.1f}")
    ax.set_ylabel('Eval Reward', color='#c9d1d9')
    ax.set_title('CTPG-MT5 — Final Per-Task Performance', color='white', fontweight='bold')
    ax.legend(); ax.grid(axis='y', alpha=0.3)
    ax.tick_params(colors='#c9d1d9')
    p4 = PLOT_DIR + '/Fig4_final_bar.png'
    plt.savefig(p4, dpi=150, bbox_inches='tight'); plt.close()
    print(f'  Saved: {p4}')

# Results CSV
rows = []
for e in eval_hist:
    row = {'step': e['step'], 'mean_reward': e['mean']}
    for i, r in enumerate(e['per_task']):
        row[f'task_{i}_g{GRAVITY_LIST[i]}'] = r
    rows.append(row)
df = pd.DataFrame(rows)
csv_path = PLOT_DIR + '/results.csv'
df.to_csv(csv_path, index=False)
print(f'  Saved: {csv_path}')
print()
print('  CTPG-MT5 Results Summary')
print(df.tail(5).to_string(index=False))

  Saved: /content/drive/MyDrive/CTPG_v2/plots/results.csv

  CTPG-MT5 Results Summary
Empty DataFrame
Columns: []
Index: []


VIDEO RECORDING (best checkpoint)    

In [ ]:
import imageio

best_ckpt = os.path.join(CKPT_DIR, 'ctpg_mt5_explicit', 'best.pt')
if os.path.exists(best_ckpt):
    best_agent, ckpt_step = CTPGAgent.load(best_ckpt)
    print(f'  Loaded best checkpoint from step {ckpt_step:,}')

    for tid in range(NUM_TASKS):
        try:
            env    = make_hc_env(GRAVITY_LIST[tid], render_mode='rgb_array')
            obs, _ = env.reset(seed=99); obs = obs.astype(np.float32)
            frames = []; total_r = 0.0
            for _ in range(1000):
                frame = env.render()
                if frame is not None:
                    frames.append(frame.astype(np.uint8))
                act = best_agent.act(obs, tid, det=True)
                obs, r, term, trunc, _ = env.step(act)
                obs = obs.astype(np.float32); total_r += r
                if term or trunc: break
            env.close()
            if frames:
                vpath = os.path.join(VID_DIR, f'task{tid}_g{GRAVITY_LIST[tid]}.mp4')
                imageio.mimsave(vpath, frames, fps=30)
                print(f'  Task {tid} g={GRAVITY_LIST[tid]}: {len(frames)} frames '
                      f'reward={total_r:.1f}  → {os.path.basename(vpath)}')
        except Exception as e:
            print(f'  Task {tid} video error: {e}')
else:
    print('  No best checkpoint found — skipping video')

ZIP ALL OUTPUTS TO DRIVE

In [ ]:
def make_zip(src, dst, exts=None):
    n = 0
    with zipfile.ZipFile(dst, 'w', zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk(src):
            for fname in files:
                if exts and not any(fname.endswith(e) for e in exts): continue
                fp = os.path.join(root, fname)
                z.write(fp, os.path.relpath(fp, src)); n += 1
    mb = os.path.getsize(dst) / 1e6
    print(f'  {os.path.basename(dst):45s} {n:4d} files  {mb:6.1f} MB')

print('Packaging outputs to Google Drive...')
make_zip(CKPT_DIR,  BASE + '/CTPG_checkpoints.zip',  ['.pt'])
make_zip(TB_DIR,    BASE + '/CTPG_tensorboard.zip')
make_zip(PLOT_DIR,  BASE + '/CTPG_plots.zip',         ['.png', '.csv'])
make_zip(LOG_DIR,   BASE + '/CTPG_logs.zip',          ['.json'])
make_zip(VID_DIR,   BASE + '/CTPG_videos.zip',        ['.mp4'])
print(f'\nAll files saved to {BASE}/')
print('Download: CTPG_checkpoints.zip | CTPG_plots.zip | CTPG_videos.zip | CTPG_logs.zip | CTPG_tensorboard.zip')
print('\nDONE.')